In [1]:
import pandas as pd
import duckdb
data = [
    # User A
    ["A", "2026-07-01 09:00:00", "O001", 100],
    ["A", "2026-07-01 10:30:00", "O002", 80],
    ["A", "2026-07-01 14:00:00", "O003", 120],
    ["A", "2026-07-02 09:20:00", "O004", 60],

    # User B
    ["B", "2026-07-01 09:20:00", "O005", 60],
    ["B", "2026-07-01 11:00:00", "O006", 90],
    ["B", "2026-07-02 16:00:00", "O007", 150],

    # User C
    ["C", "2026-07-01 10:00:00", "O008", 200],
    ["C", "2026-07-01 15:30:00", "O009", 50],
    ["C", "2026-07-02 12:00:00", "O010", 75],
]

df = pd.DataFrame(
    data,
    columns=["user_id", "order_time", "order_id", "amount"]
)

df["order_time"] = pd.to_datetime(df["order_time"])

print(df)



  user_id          order_time order_id  amount
0       A 2026-07-01 09:00:00     O001     100
1       A 2026-07-01 10:30:00     O002      80
2       A 2026-07-01 14:00:00     O003     120
3       A 2026-07-02 09:20:00     O004      60
4       B 2026-07-01 09:20:00     O005      60
5       B 2026-07-01 11:00:00     O006      90
6       B 2026-07-02 16:00:00     O007     150
7       C 2026-07-01 10:00:00     O008     200
8       C 2026-07-01 15:30:00     O009      50
9       C 2026-07-02 12:00:00     O010      75


# 题目要求

## 分别使用 SQL 和 Pandas 完成：

- 计算每个用户每次下单后，截至当前订单的平均消费金额。

- 最终输出字段：

`user_id`
`order_time`
`order_id`
`amount`
`running_avg_amount`

In [5]:
# SQL轨道

query = """

SELECT
    user_id,
    order_time,
    order_id,
    amount,
    AVG(amount)
        OVER(
            PARTITION BY user_id ORDER BY order_time,order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )::INTEGER AS running_avg_amount
FROM df
ORDER BY user_id,order_time,order_id
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,user_id,order_time,order_id,amount,running_avg_amount
0,A,2026-07-01 09:00:00,O001,100,100
1,A,2026-07-01 10:30:00,O002,80,90
2,A,2026-07-01 14:00:00,O003,120,100
3,A,2026-07-02 09:20:00,O004,60,90
4,B,2026-07-01 09:20:00,O005,60,60
5,B,2026-07-01 11:00:00,O006,90,75
6,B,2026-07-02 16:00:00,O007,150,100
7,C,2026-07-01 10:00:00,O008,200,200
8,C,2026-07-01 15:30:00,O009,50,125
9,C,2026-07-02 12:00:00,O010,75,108


In [ ]:
# PANDAS轨道1

df_pd1 = (
    df
    .sort_values(by=['user_id','order_time','order_id'],ascending=[True,True,True])
    .assign(
        running_total_sum = lambda x:(
            x.groupby('user_id')['amount'].cumsum()
        ),
        running_total_count = lambda x:(
            x.groupby('user_id').cumcount()+1
        ),
        running_avg_amount = lambda x:x['running_total_sum'] / x['running_total_count']
    )  
)



,user_id,order_time,order_id,amount,running_total_sum,running_total_count,running_avg_amount
0,A,2026-07-01 09:00:00,O001,100,100,1,100.000000
1,A,2026-07-01 10:30:00,O002,80,180,2,90.000000
2,A,2026-07-01 14:00:00,O003,120,300,3,100.000000
3,A,2026-07-02 09:20:00,O004,60,360,4,90.000000
4,B,2026-07-01 09:20:00,O005,60,60,1,60.000000
5,B,2026-07-01 11:00:00,O006,90,150,2,75.000000
6,B,2026-07-02 16:00:00,O007,150,300,3,100.000000
7,C,2026-07-01 10:00:00,O008,200,200,1,200.000000
8,C,2026-07-01 15:30:00,O009,50,250,2,125.000000
9,C,2026-07-02 12:00:00,O010,75,325,3,108.333333


In [16]:
df_pd2 = (
    df
    .sort_values(by=['user_id', 'order_time', 'order_id'])
    .assign(
        running_avg_amount=lambda x: (
            x.groupby('user_id')['amount']
             .expanding()
             .mean()
             .reset_index(level=0, drop=True)
        )
    )
    .reset_index(drop=True)
)
df_pd2

,user_id,order_time,order_id,amount,running_avg_amount
0,A,2026-07-01 09:00:00,O001,100,100.000000
1,A,2026-07-01 10:30:00,O002,80,90.000000
2,A,2026-07-01 14:00:00,O003,120,100.000000
3,A,2026-07-02 09:20:00,O004,60,90.000000
4,B,2026-07-01 09:20:00,O005,60,60.000000
5,B,2026-07-01 11:00:00,O006,90,75.000000
6,B,2026-07-02 16:00:00,O007,150,100.000000
7,C,2026-07-01 10:00:00,O008,200,200.000000
8,C,2026-07-01 15:30:00,O009,50,125.000000
9,C,2026-07-02 12:00:00,O010,75,108.333333
